In [1]:
from pyc8y.app import connect
c8y = await connect()

Please enter the Cumulocity base URL or hostname: https://pmt-training.eu-latest.cumulocity.com/
Please enter the Cumulocity tenant ID: t21106993
Please enter the Cumulocity username: christoph.souris@cumulocity.com
Please enter the Cumulocity password: ········


In [2]:
mos = await c8y.inventory.get_all(limit=1000, workers=10)

In [3]:
len(mos)

1000

In [4]:
ids = [m.id for m in mos]

In [31]:
from pyc8y.model import Device
device = await Device(c8y, name="TestDevice", type="c8y_TestDevice").create()

In [34]:
from pyc8y.model import Measurement
from datetime import datetime, timedelta

start = datetime.fromisoformat("2020-01-01T00:00:00Z") 
delta = timedelta(seconds=30)

# create measurements
series_a = [Measurement(type="c8y_Test", source=device.id, time=start + (delta*i),  series=("c8y_Test.series_a", i, "#")) for i in range(120)]
series_b = [Measurement(type="c8y_Test", source=device.id, time=start + (delta*i*10), series=("c8y_Test.series_b", i, "#")) for i in range(12)]


In [36]:
await c8y.measurements.create(*series_a, *series_b, workers=10)

In [37]:
n = await c8y.measurements.get_all(source=device.id)

In [39]:
len(n)

133

In [54]:
await c8y.measurements.get_series(
    source=device.id,
    series=["c8y_Test.series_b", "c8y_Test.series_a"],
    date_from="2020-01-01",
    date_to="now",
    aggregation_function=["min", "max", "count"],
    aggregation_interval="1m",
    timestamps=True,
)

{'values': {'2020-01-01T00:00:00.000Z': [{'min': 0.0,
    'max': 0.0,
    'count': 4.0},
   {'min': 0.0, 'max': 1.0, 'count': 4.0}],
  '2020-01-01T00:01:00.000Z': [{'count': 2.0},
   {'min': 2.0, 'max': 3.0, 'count': 2.0}],
  '2020-01-01T00:02:00.000Z': [{'count': 2.0},
   {'min': 4.0, 'max': 5.0, 'count': 2.0}],
  '2020-01-01T00:03:00.000Z': [{'count': 2.0},
   {'min': 6.0, 'max': 7.0, 'count': 2.0}],
  '2020-01-01T00:04:00.000Z': [{'count': 2.0},
   {'min': 8.0, 'max': 9.0, 'count': 2.0}],
  '2020-01-01T00:05:00.000Z': [{'min': 1.0, 'max': 1.0, 'count': 3.0},
   {'min': 10.0, 'max': 11.0, 'count': 3.0}],
  '2020-01-01T00:06:00.000Z': [{'count': 2.0},
   {'min': 12.0, 'max': 13.0, 'count': 2.0}],
  '2020-01-01T00:07:00.000Z': [{'count': 2.0},
   {'min': 14.0, 'max': 15.0, 'count': 2.0}],
  '2020-01-01T00:08:00.000Z': [{'count': 2.0},
   {'min': 16.0, 'max': 17.0, 'count': 2.0}],
  '2020-01-01T00:09:00.000Z': [{'count': 2.0},
   {'min': 18.0, 'max': 19.0, 'count': 2.0}],
  '2020-01-01T

In [59]:
import orjson
[orjson.dumps(x.to_json()) for x in series_a[0:13]]

[b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:00:00.000Z","c8y_Test":{"series_a":{"value":0,"unit":"#"}}}',
 b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:00:30.000Z","c8y_Test":{"series_a":{"value":1,"unit":"#"}}}',
 b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:01:00.000Z","c8y_Test":{"series_a":{"value":2,"unit":"#"}}}',
 b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:01:30.000Z","c8y_Test":{"series_a":{"value":3,"unit":"#"}}}',
 b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:02:00.000Z","c8y_Test":{"series_a":{"value":4,"unit":"#"}}}',
 b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:02:30.000Z","c8y_Test":{"series_a":{"value":5,"unit":"#"}}}',
 b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:03:00.000Z","c8y_Test":{"series_a":{"value":6,"unit":"#"}}}',
 b'{"type":"c8y_Test","source":{"id":"5379946991"},"tim

In [61]:
import orjson
[orjson.dumps(x.to_json()) for x in series_b[0:2]]

[b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:00:00.000Z","c8y_Test":{"series_b":{"value":0,"unit":"#"}}}',
 b'{"type":"c8y_Test","source":{"id":"5379946991"},"time":"2020-01-01T00:05:00.000Z","c8y_Test":{"series_b":{"value":1,"unit":"#"}}}']

In [62]:
await c8y.measurements.create(*series_a[0:13], *series_b[0:2])